# Arsip versi data nyata

Versi sebelumnya, dijalankan pada 57.362 komentar asli. Notebook utama sekarang memakai satu contoh sintetis per metode. Jalankan arsip ini dari folder legacy bila ingin mengulang alur data nyata.

# Preprocessing teks Indonesia / MBG — langkah demi langkah

**Baca dari atas ke bawah. Setiap tahap = satu fungsi pendek → satu pemanggilan → before/after.**
Semua fungsi tertanam di notebook, tanpa import file proyek. Ubah `INPUT` dan `TEXT` lalu **Run All**.
Input bawaan: **57.362 komentar asli TikTok MBG**, kolom `comment`, bukan `clean_comment`.
Seluruh baris diproses, tanpa sampling. Tidak membutuhkan internet/GPU.

| Bagian | Isi | Hasil |
|---|---|---|
| A | Input, tipe data, encoding, Unicode, HTML, karakter tak terlihat, spasi | `text_safe` |
| B | Teks tersamarkan, URL, email, telepon, mention, hashtag, lowercase | awal normalisasi |
| C | Slang, elongation, tawa, reduplikasi, daerah, typo, tanda baca berulang | `text_normalized` |
| D | Emoji/emoticon, angka opsional, tanda baca, repetisi kata, stopword, stemming | `text_clean` |
| E | Alternatif deskripsi emoji, resep IndoBERTweet, tokenisasi | versi tambahan |
| F | Perbandingan seluruh tahap, ekspor, fungsi sekali panggil, verifikasi | hasil siap dipakai |

**“Bersih maksimal” tersedia di `text_clean`.** Safe dan normalized juga disimpan sehingga
efek penghapusan dapat dibandingkan. Angka tidak dibuang secara default karena nominal/tanggal
bisa penting; ubah `REMOVE_NUMBERS=True` untuk view lebih agresif. Kata negasi serta akronim
MBG dilindungi. Stemming/stopword aktif hanya di cabang clean.

Fokus notebook ini **teks saja**. Tidak ada training, embedding, deteksi bot, atau penghapusan baris.
Raw, metadata, label, urutan, dan ID input dipertahankan. Skor model belum diukur.

## Persiapan · dependensi dan konfigurasi

Instal sekali saat online: `python -m pip install -r requirements.txt`.
Untuk versi lengkap di notebook ini diperlukan `pandas`, `ftfy`, `emoji`, `beautifulsoup4`, dan `Sastrawi`.
Kamus GitHub dapat disiapkan sekali dan dibaca offline. Tidak ada unduhan otomatis.
CSV dibaca sebagai string agar ID/nol awal/angka teks tidak berubah karena inferensi tipe;
untuk Parquet/JSONL tipe sumber tetap dibaca dan nilai non-string ditandai pada validasi.

In [1]:
"""Text-only preprocessing in small deterministic steps; also embedded in the notebook.

API: preprocess_texts(series, slang=None, options=None). Input order/index are preserved.
No downloads, no fitting, no record deletion. Each stage can be copied independently.
"""
import html
import re
import unicodedata
from functools import lru_cache

import emoji
import ftfy
import pandas as pd
from pathlib import Path
import hashlib
import json
import time
from IPython.display import display

In [2]:
INPUT = Path('../data/MBG_PROCESSED.csv')  # ganti untuk data baru
TEXT = 'comment'
SEP, ENCODING = ',', 'utf-8-sig'
OUT = Path('../hasil_stepwise')
RESOURCE_DIR = Path('../resources/downloaded')
DOWNLOAD_RESOURCES = False  # True sekali online; False untuk penggunaan offline
REVIEWED_LEXICON = Path('reviewed_lexicon.csv')
HTML_INPUT = False  # True hanya jika sumber memang markup HTML
DECODE_ESCAPED_BYTES = False  # True jika \xf0... benar-benar emoji ter-escape
NORMALIZE_REGIONAL = False  # jangan otomatis menerjemahkan semua bahasa daerah
REMOVE_NUMBERS = False
REMOVE_STOPWORDS, STEMMING = True, True
TYPO_APPROVED = {}  # setelah review: {'maknan': 'makanan', 'bergzi': 'bergizi'}
HASHTAG_APPROVED = {}  # contoh: {'tolakmbg': 'tolak mbg'}
PREVIEW_ROWS = 8
SHOW_RULE_EXAMPLES = False  # True hanya untuk contoh aturan buatan, bukan hasil dataset
pd.set_option('display.max_colwidth', 120)

In [3]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
STOPWORDS = set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split()) - NEGATIONS - PROTECTED
REGIONAL = {'ora': 'tidak', 'teu': 'tidak', 'indak': 'tidak', 'apik': 'bagus'}
OBFUSCATION = {'k3r4cunan': 'keracunan', 'k3r4cun4n': 'keracunan',
               'ker@cunan': 'keracunan', 'koru*si': 'korupsi'}
# URL/email/account/hashtag are single tokens: normalization never enters them.
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)


def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

## A · Input dan cleaning teknis

Belum lowercase, belum buang emoji/angka/tanda baca. Teks raw tidak ditimpa.
Helper `run_step` hanya mengurangi kode berulang untuk pemrosesan kata unik dan tampilan:
fungsi preprocessing pada setiap sel tetap bisa disalin dan dipanggil langsung.

In [4]:
def load_data(path):
    path = Path(path)
    if path.suffix.lower() in {'.csv', '.tsv'}:
        return pd.read_csv(path, sep='\t' if path.suffix.lower() == '.tsv' else SEP,
                           encoding=ENCODING, dtype='string', keep_default_na=False, na_values=[''])
    if path.suffix.lower() in {'.parquet', '.pq'}:
        return pd.read_parquet(path)
    if path.suffix.lower() in {'.jsonl', '.ndjson'}:
        return pd.read_json(path, lines=True)
    raise ValueError('Gunakan CSV/TSV/Parquet/JSONL.')

if INPUT is None:
    raise ValueError('Isi INPUT dengan path dataset asli. Tidak ada fallback data sintetis.')
df = load_data(INPUT)
print('Sumber data:', INPUT.resolve())
print('SHA-256:', hashlib.sha256(INPUT.read_bytes()).hexdigest())
print('Kolom teks:', TEXT, '| seluruh baris digunakan; tanpa sampling')
print('Label di file adalah keluaran model; tidak dipakai untuk preprocessing/training.')
if TEXT not in df:
    raise KeyError(f'Kolom {TEXT!r} tidak ditemukan: {list(df)}')
raw = df[TEXT].copy()
texts = raw.copy()
history, audit = {}, []
print('Baris:', len(df), '| kolom:', list(df))
display(df.head(PREVIEW_ROWS))

Sumber data: D:\Lomba\2026\Gemastik\Notebook\Preprocessing_universal\data\MBG_PROCESSED.csv
SHA-256: d8d1da42d483975a8a4140d9babfc0f759e07e96ad6ff4482bdc38b3ea2b659b
Kolom teks: comment | seluruh baris digunakan; tanpa sampling
Label di file adalah keluaran model; tidak dipakai untuk preprocessing/training.
Baris: 57362 | kolom: ['video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname', 'comment', 'create_time', 'clean_comment', 'is_empty_after_cleaning', 'sentiment', 'sentiment_score', 'agufsamudra_sentiment', 'agufsamudra_sentiment_score', 'mdhugol_sentiment', 'mdhugol_sentiment_score', 'aardiiiiy_sentiment', 'aardiiiiy_sentiment_score', 'final_sentiment_voting']


,video_id,comment_id,parent_comment_id,level,username,nickname,comment,create_time,clean_comment,is_empty_after_cleaning,sentiment,sentiment_score,agufsamudra_sentiment,agufsamudra_sentiment_score,mdhugol_sentiment,mdhugol_sentiment_score,aardiiiiy_sentiment,aardiiiiy_sentiment_score,final_sentiment_voting
0,7553950099299110151,7554042072476943111,<NA>,0.0,aris.jo2,Aris Jo,"ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E",2025-09-25 22:01:26,ibuk ngomongnya times new roman bold caplock nyala 16 ctrl e,False,netral,0.9754078388214111,negatif,0.9957045912742615,netral,0.8462600111961365,negatif,0.9895619750022888,negatif
1,7553950099299110151,7554050261040661266,7.554042072476943e+18,1.0,heztyannadrwskinc,Hezty Anna DrwSkinca,ENTER ENTER ENTER Langsung ke INTI nya,2025-09-25 22:33:09,enter enter enter langsung ke inti nya,False,netral,0.9549044966697693,negatif,0.9457594156265259,netral,0.9876952171325684,netral,0.9482853412628174,netral
2,7553950099299110151,7554184392182055687,7.554042072476943e+18,1.0,royantek,Roy Ahmad,SHIFT+ENTER langsung ke lembar berikutnya🗿,2025-09-26 07:13:58,shift enter langsung ke lembar berikutnya,False,netral,0.974612295627594,negatif,0.9700665473937988,netral,0.9947147965431213,netral,0.9880483746528625,netral
3,7553950099299110151,7554145436236776208,7.554042072476943e+18,1.0,bitas__,bitas__,@Lfajeriah,2025-09-26 04:42:55,<NA>,True,empty,0.0,empty,0.0,empty,0.0,empty,0.0,empty
4,7553950099299110151,7554951123045516049,7.554042072476943e+18,1.0,shiva.slv_,⋆𐙚｡Shiva.ᡣ𐭩˚˚,🤣🤣🤣😭😭,2025-09-28 08:49:05,<NA>,True,empty,0.0,empty,0.0,empty,0.0,empty,0.0,empty
5,7553950099299110151,7554791824016818961,7.554042072476943e+18,1.0,alevayn,Deva♓💜♋,🤣🤣,2025-09-27 22:30:55,<NA>,True,empty,0.0,empty,0.0,empty,0.0,empty,0.0,empty
6,7553950099299110151,7555800499125093138,7.554042072476943e+18,1.0,syariefyunitaa,Yunita Syarief,😭😭Ngakak,2025-09-30 15:45:03,ngakak,False,negatif,0.9961302280426025,negatif,0.9419488906860352,positif,0.9848239421844482,negatif,0.8675886392593384,negatif
7,7553950099299110151,7554076193713521426,7.554042072476943e+18,1.0,myenisa,myenisa,"sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan",2025-09-26 00:13:50,sepertinya tombolnya bukan enter kak tapi entar jadi agak delay ke tujuan,False,negatif,0.5387620329856873,negatif,0.9805276393890381,netral,0.5243319272994995,netral,0.9845805764198303,negatif


In [5]:
def visible(value):
    return repr(value)  # newline, spasi, null, dan tipe angka terlihat jelas

def run_step(before, function, label, enabled=True):
    started = time.perf_counter()
    if enabled:
        # valid_text is handled per-row: input may be list/null, not hashable strings.
        if function.__name__ == 'valid_text':
            after = before.map(function)
        else:
            lookup = {s: function(s) for s in pd.unique(before)}
            after = before.map(lookup)
    else:
        after = before.copy()
    table = pd.DataFrame({'before': before.map(visible), 'after': after.map(visible)})
    changed = table.before.ne(table.after)
    audit.append(dict(stage=label, enabled=bool(enabled), changed=int(changed.sum()),
                      empty=int(after.eq('').sum()), seconds=round(time.perf_counter()-started, 4)))
    history[label] = after.copy()
    print(f'{label}: aktif={enabled}; berubah={changed.sum():,}/{len(before):,}; kosong={after.eq("").sum():,}')
    # Changed rows first; retain order within each set. Empty input still works.
    order = sorted(range(len(table)), key=lambda i: not changed.iloc[i])
    display(table.iloc[order].head(PREVIEW_ROWS))
    return after

def show_example(example, function):
    display(pd.DataFrame({'contoh_before': [visible(example)],
                          'contoh_after': [visible(function(example))]}))

### A1 · Validasi tipe teks

Null/angka/list menjadi string kosong pada teks kerja, bukan literal `nan` atau `123`. String asli `"nan"`/`"None"` tetap teks. Baris tidak dibuang; nilai sumber tetap ada di raw.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [6]:
def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

In [7]:
texts = run_step(texts, valid_text, 'A1 · Validasi tipe teks', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example(None, valid_text)

A1 · Validasi tipe teks: aktif=True; berubah=485/57,362; kosong=485


,before,after
529,<NA>,''
873,<NA>,''
1012,<NA>,''
1097,<NA>,''
1149,<NA>,''
1783,<NA>,''
1793,<NA>,''
1830,<NA>,''


In [8]:
validation = pd.DataFrame({'missing': raw.isna(),
    'non_string': raw.map(lambda x: not isinstance(x, str)) & ~raw.isna(),
    'empty': texts.eq('')})
display(validation.sum().to_frame('jumlah'))

,jumlah
missing,485
non_string,0
empty,485


### A2 · Decode byte escape

Opsional untuk hasil scraping dengan byte UTF-8 tersimpan literal. Byte invalid tetap utuh. Bukan decode `unicode_escape` pada seluruh teks, yang bisa merusak Unicode valid.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [9]:
def decode_byte_escapes(text):
    def decode(match):
        try:
            return bytes.fromhex(match[0].replace('\\x', '')).decode('utf-8')
        except UnicodeDecodeError:
            return match[0]  # invalid bytes remain visible for review
    return re.sub(r'(?:\\x[0-9a-fA-F]{2})+', decode, text)

In [10]:
texts = run_step(texts, decode_byte_escapes, 'A2 · Decode byte escape', enabled=DECODE_ESCAPED_BYTES)
if SHOW_RULE_EXAMPLES:
    show_example('emoji \\xf0\\x9f\\x98\\xa1', decode_byte_escapes)

A2 · Decode byte escape: aktif=False; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### A3 · Perbaiki encoding rusak

ftfy memperbaiki mojibake. Perubahan tipografi, lebar karakter, dan HTML di ftfy dimatikan agar setiap transformasi terlihat pada tahapnya sendiri.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [11]:
def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

In [12]:
texts = run_step(texts, repair_encoding, 'A3 · Perbaiki encoding rusak', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('cafÃ©, nasi sehat', repair_encoding)

A3 · Perbaiki encoding rusak: aktif=True; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### A4 · Normalisasi Unicode NFC

Samakan representasi kanonis. Tidak menghapus aksara daerah/emoji dan tidak memaksa ASCII/NFKC.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [13]:
def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

In [14]:
texts = run_step(texts, normalize_unicode, 'A4 · Normalisasi Unicode NFC', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('Café', normalize_unicode)

A4 · Normalisasi Unicode NFC: aktif=True; berubah=2/57,362; kosong=485


,before,after
31963,'@ɑׁׅ݊ꪀ݊ꪀꪱׁׁׁׅׅׅƙׁׅɑׁ 囧 @andinnnn @🐣...','@ɑׁׅ݊ꪀ݊ꪀꪱׁׁׁׅׅׅƙׁׅɑׁ 囧 @andinnnn @🐣...'
56126,'@ɑׁׅ݊ꪀ݊ꪀꪱׁׁׁׅׅׅƙׁׅɑׁ 囧 @andinnnn @🐣...','@ɑׁׅ݊ꪀ݊ꪀꪱׁׁׁׅׅׅƙׁׅɑׁ 囧 @andinnnn @🐣...'
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'


### A5 · Buang markup HTML bila ada

Parser HTML hanya untuk sumber yang benar-benar markup. Plain text `nilai < 5 > 1` jangan diparse sembarangan. Script/style dibuang. Contoh kecil tetap menunjukkan parser walaupun tahap dataset dimatikan.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [15]:
def remove_html(text):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    return soup.get_text(' ')

In [16]:
texts = run_step(texts, remove_html, 'A5 · Buang markup HTML bila ada', enabled=HTML_INPUT)
if SHOW_RULE_EXAMPLES:
    show_example('<p>Menu <b>sehat</b></p><script>bad()</script>', remove_html)

A5 · Buang markup HTML bila ada: aktif=False; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### A6 · Decode HTML entities

`&amp;`, `&quot;`, `&#...;` menjadi karakter aslinya. Satu lapis decoding; tidak berulang tanpa batas pada teks yang sengaja menuliskan entity.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [17]:
def decode_html_entities(text):
    return html.unescape(text)

In [18]:
texts = run_step(texts, decode_html_entities, 'A6 · Decode HTML entities', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('MBG &amp; gizi &quot;baik&quot;', decode_html_entities)

A6 · Decode HTML entities: aktif=True; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### A7 · Bersihkan karakter kontrol

Hapus BOM, zero-width space, word joiner, null byte; kontrol lain menjadi spasi. ZWJ/ZWNJ dan variation selectors dipertahankan agar emoji gabungan tidak pecah.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [19]:
def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    # ZWJ, ZWNJ and variation selectors remain: preserve emoji and valid writing.

In [20]:
texts = run_step(texts, remove_invisible, 'A7 · Bersihkan karakter kontrol', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('teks\u200bbaru\x00 👩\u200d🍳', remove_invisible)

A7 · Bersihkan karakter kontrol: aktif=True; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### A8 · Rapikan whitespace

Gabungkan tab/newline/spasi berulang dan strip tepi. Notebook ini untuk komentar satu baris; untuk dokumen berparagraf, pakai versi yang mempertahankan newline.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [21]:
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [22]:
texts = run_step(texts, normalize_whitespace, 'A8 · Rapikan whitespace', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('  menu\tsehat\r\n  merata  ', normalize_whitespace)

A8 · Rapikan whitespace: aktif=True; berubah=2,888/57,362; kosong=485


,before,after
142,'CTRL + ALT + DEL\n😂','CTRL + ALT + DEL 😂'
219,'😭😭😭😭😭 @c @Mel vibes bunda mande n the aniezzzzz😭','😭😭😭😭😭 @c @Mel vibes bunda mande n the aniezzzzz😭'
277,'@intanᏊ˘\xa0ꈊ\xa0˘\xa0Ꮚ 🤣','@intanᏊ˘ ꈊ ˘ Ꮚ 🤣'
584,"'karena vt ini pernah coba cari di website, trus bikin sendiri.... enak banget 🥰\njadi pengen coba menu2 nusantara l...","'karena vt ini pernah coba cari di website, trus bikin sendiri.... enak banget 🥰 jadi pengen coba menu2 nusantara la..."
642,"'mbg anak ku kmren crita nya sandwich tapi isi nya\nroti tawar 1lemabar,ayam suir 6suiran,selada timun tomat, saos c...","'mbg anak ku kmren crita nya sandwich tapi isi nya roti tawar 1lemabar,ayam suir 6suiran,selada timun tomat, saos ca..."
662,"'ya lebih baik ga diiyakan lah selama itu makanan sehat dan begizi. burger, spagetti, bakmi, kacang sukro\napanya yg...","'ya lebih baik ga diiyakan lah selama itu makanan sehat dan begizi. burger, spagetti, bakmi, kacang sukro apanya yg ..."
665,'pendukung fanatik Prabowo anomali 78 ini udah ga ketolong yaa😭 bisa2 nya ada orang yg peduli buat kritik gizi denga...,'pendukung fanatik Prabowo anomali 78 ini udah ga ketolong yaa😭 bisa2 nya ada orang yg peduli buat kritik gizi denga...
689,"'jngn dengerkn permintaan anak""\nada ny program ini kan buat GIZI bukn buat kesenangan.\nyg ngmng ini dokter ahli GI...","'jngn dengerkn permintaan anak"" ada ny program ini kan buat GIZI bukn buat kesenangan. yg ngmng ini dokter ahli GIZI..."


In [23]:
text_safe = texts.copy()  # snapshot cleaning teknis, belum normalisasi leksikal

## B · Elemen khusus media sosial

Mulai cabang normalisasi. Placeholder `URL/EMAIL/PHONE/USER` menyamakan identitas acak tanpa
mengosongkan kalimat. Setelah lowercase token menjadi huruf kecil. Ini transformasi teks,
bukan jaminan anonimisasi. Sumber asli tetap di raw.

### B1 · Normalisasi teks tersamarkan

Hanya allowlist kata yang eksplisit: tidak mengubah semua 3→e/4→a karena nomor dan kode bisa rusak. Dilakukan sebelum mention supaya `ker@cunan` bisa dikenali sebagai kata.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [24]:
def normalize_obfuscation(text, mapping=OBFUSCATION):
    return re.sub(r'(?<!\w)[a-zA-Z][a-zA-Z0-9@*$]{3,}(?!\w)',
                  lambda m: mapping.get(m[0].casefold(), m[0]), text)

In [25]:
texts = run_step(texts, normalize_obfuscation, 'B1 · Normalisasi teks tersamarkan', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('k3r4cUn4n ker@cunan koru*si S2 Rp10.000', normalize_obfuscation)

B1 · Normalisasi teks tersamarkan: aktif=True; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### B2 · Ganti URL

URL panjang menjadi placeholder. Tanda baca penutup dipertahankan dengan heuristik sederhana; raw tetap menyimpan tautan lengkap.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [26]:
def replace_urls(text):
    # Preserve terminal sentence punctuation; a heuristic, not a complete URL parser.
    return URL.sub(lambda m: ' URL ' + m[0][len(m[0].rstrip('.,!?;)')):], text)

In [27]:
texts = run_step(texts, replace_urls, 'B2 · Ganti URL', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('cek https://contoh.id/x?q=gk, sekarang!', replace_urls)

B2 · Ganti URL: aktif=True; berubah=22/57,362; kosong=485


,before,after
3753,'https://vt.tiktok.com/ZSyQTyC5y/',' URL '
6718,'https://vm.tiktok.com/ZMA5bQuFn/',' URL '
10167,'https://www.teropongmalut.com/pulau-fao-digunduli-tambang-izin-siapa-untuk-siapa/ #SavePulauFao',' URL #SavePulauFao'
10864,"'Bongkar tuntas MBG, sila cek disini https://youtu.be/DLM0lZFOzQk?si=FXmHLT-QVmYXSUZR'","'Bongkar tuntas MBG, sila cek disini URL '"
13818,'jauhhhlh kl di blg dr 4 sehat 5 sempurna....hadewwww...diluar program😅','jauhhhlh kl di blg dr 4 sehat 5 sempurna....hadew URL program😅'
15504,'https://www.instagram.com/reel/DPC3qmVknrQ/?igsh=ZXoxdnN3YWUwb2Rs',' URL '
17997,'https://id.shp.ee/wje2pu8?smtt=0.0.9',' URL '
18144,"'nah nah.. itu dia, klo DPR udh mulai berbicara angka2 itu dia celah korupsi mereka hadeeewwwww.. nista betuuul😥'","'nah nah.. itu dia, klo DPR udh mulai berbicara angka2 itu dia celah korupsi mereka hadeeeww URL .. nista betuuul😥'"


### B3 · Ganti alamat email

Email sebelum mention agar `@domain` tidak dianggap akun.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [28]:
def replace_emails(text):
    return re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', ' EMAIL ', text)

In [29]:
texts = run_step(texts, replace_emails, 'B3 · Ganti alamat email', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('kirim admin@contoh.id ya', replace_emails)

B3 · Ganti alamat email: aktif=True; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','SHIFT+ENTER langsung ke lembar berikutnya🗿'
3,'@Lfajeriah','@Lfajeriah'
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭Ngakak','😭😭Ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'"


### B4 · Ganti nomor telepon

Pola konservatif nomor seluler Indonesia. Nominal/tanggal tidak diganti. Format telepon di luar pola perlu aturan tambahan.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [30]:
def replace_phones(text):
    return re.sub(r'(?<!\w)(?:\+62|62|0)[ -]?8(?:[ -]?\d){8,11}(?!\w)', ' PHONE ', text)

In [31]:
texts = run_step(texts, replace_phones, 'B4 · Ganti nomor telepon', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('WA +62 812-3456-7890, harga Rp10.000', replace_phones)

B4 · Ganti nomor telepon: aktif=True; berubah=6/57,362; kosong=485


,before,after
12676,'Yang butuh loker bisa hubungi 083153023781','Yang butuh loker bisa hubungi PHONE '
25097,'Siap suplay Bandeng Presto untuk kebutuhan Dapur MBG isi 2-3-4-5/Kotak (bisa request) Harga 8500/Kotak belum termas...,'Siap suplay Bandeng Presto untuk kebutuhan Dapur MBG isi 2-3-4-5/Kotak (bisa request) Harga 8500/Kotak belum termas...
25999,'Foodtray 5 Sekat Bahan : stainless 304 Embos : sus304 Diameter : 28x22x6 Wa : 087868301972 kita juga ready perlengk...,'Foodtray 5 Sekat Bahan : stainless 304 Embos : sus304 Diameter : 28x22x6 Wa : PHONE kita juga ready perlengkapan ...
40868,'Yang butuh loker bisa hubungi 083153023781','Yang butuh loker bisa hubungi PHONE '
52175,'Siap suplay Bandeng Presto untuk kebutuhan Dapur MBG isi 2-3-4-5/Kotak (bisa request) Harga 8500/Kotak belum termas...,'Siap suplay Bandeng Presto untuk kebutuhan Dapur MBG isi 2-3-4-5/Kotak (bisa request) Harga 8500/Kotak belum termas...
52505,'Foodtray 5 Sekat Bahan : stainless 304 Embos : sus304 Diameter : 28x22x6 Wa : 087868301972 kita juga ready perlengk...,'Foodtray 5 Sekat Bahan : stainless 304 Embos : sus304 Diameter : 28x22x6 Wa : PHONE kita juga ready perlengkapan ...
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','ENTER ENTER ENTER Langsung ke INTI nya'


### B5 · Ganti mention

Mention menjadi placeholder. Nama orang di kalimat biasa tetap ada.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [32]:
def replace_mentions(text):
    return re.sub(r'(?<![\w@])@[\w.]+', ' USER ', text)

In [33]:
texts = run_step(texts, replace_mentions, 'B5 · Ganti mention', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('@budi dan @siti.id bahas MBG', replace_mentions)

B5 · Ganti mention: aktif=True; berubah=2,051/57,362; kosong=485


,before,after
3,'@Lfajeriah',' USER '
10,'@eun.sang @itsprilllyyyy',' USER USER '
13,'@alesha 😭',' USER 😭'
15,'@jesiaa bella',' USER bella'
22,'@Uriel @helena 😭',' USER USER 😭'
24,'😭😭 @dindaaja','😭😭 USER '
29,'@r @najwa @Regiiiiinnaa_ 😭',' USER USER USER 😭'
32,'@uky 😭',' USER 😭'


### B6 · Segmentasi hashtag

Pisahkan CamelCase/akronim, pertahankan isi kata. Hashtag lowercase gabungan hanya dipecah lewat mapping yang disetujui.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [34]:
def segment_hashtags(text, mapping=None):
    mapping = {} if mapping is None else mapping
    def split(match):
        word = match[1]
        # #tolakmbg stays tolakmbg unless an explicit reviewed map is supplied.
        return ' ' + mapping.get(word.casefold(), re.sub(
            r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word)) + ' '
    return re.sub(r'(?<!\w)#(\w+)', split, text)

In [35]:
texts = run_step(texts, lambda s: segment_hashtags(s, HASHTAG_APPROVED), 'B6 · Segmentasi hashtag', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('#TolakMBG #DukungProgramMBG #tolakmbg', lambda s: segment_hashtags(s, HASHTAG_APPROVED))

B6 · Segmentasi hashtag: aktif=True; berubah=56/57,362; kosong=485


,before,after
4527,'#CopotBGNdanJajarannya',' Copot BG Ndan Jajarannya '
6557,"'di desaku ada banyak ahli gizi universitas ternama di indonesia, tapi tidak di pakai semua karena yang di pakek ada...","'di desaku ada banyak ahli gizi universitas ternama di indonesia, tapi tidak di pakai semua karena yang di pakek ada..."
6610,'mbg : makanan beracun gratis!!! #LOLL','mbg : makanan beracun gratis!!! LOLL '
6842,'#FREEPALESTINE🇵🇸 #FREEPALESTINE🇵🇸 #FREEPALESTINE🇵🇸 #SAVEPALESTINE🇵🇸 #SAVEPALESTINE🇵🇸 #SAVEPALESTINE🇵🇸',' FREEPALESTINE 🇵🇸 FREEPALESTINE 🇵🇸 FREEPALESTINE 🇵🇸 SAVEPALESTINE 🇵🇸 SAVEPALESTINE 🇵🇸 SAVEPALESTINE 🇵🇸'
7300,"'MBG anakq malah kentang goreng,apel kecil,sama susu kacang hijau yg 2000an itu untuk menu kering🤣🤣dan menu makannya...","'MBG anakq malah kentang goreng,apel kecil,sama susu kacang hijau yg 2000an itu untuk menu kering🤣🤣dan menu makannya..."
7328,'#hentikanMBG',' hentikan MBG '
7341,"'STOP MBG, #stopmbg (ramaikan tagar ini)'","'STOP MBG, stopmbg (ramaikan tagar ini)'"
7737,'By data 👍 #MBG 🇮🇩','By data 👍 MBG 🇮🇩'


### B7 · Lowercase

Casefold untuk cabang normalisasi/clean. Kapitalisasi asli tetap tersedia di safe/raw untuk model yang membutuhkannya.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [36]:
def lowercase(text):
    return text.casefold()

In [37]:
texts = run_step(texts, lowercase, 'B7 · Lowercase', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('TIDAK Bagus! SPPG MBG', lowercase)

B7 · Lowercase: aktif=True; berubah=24,175/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','enter enter enter langsung ke inti nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','shift+enter langsung ke lembar berikutnya🗿'
3,' USER ',' user '
6,'😭😭Ngakak','😭😭ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'"
10,' USER USER ',' user user '
13,' USER 😭',' user 😭'


## C · Normalisasi kata dan ekspresi

Tidak ada fuzzy replacement global. Slang memakai batas kata utuh; kata ambigu/akronim
dilindungi. Kamus eksternal menjadi kandidat review, bukan aturan otomatis yang menimpa makna.

### C0 · Kamus lokal dan tiga sumber GitHub

Colloquial, Kamus Alay, dan Combined Slang dibaca lokal bila sudah diunduh.
Semua mapping eksternal berstatus pending, konflik tidak diterapkan. Untuk pemakaian ulang,
isi `review_status=approved` pada pasangan yang sudah dicek lalu simpan `reviewed_lexicon.csv`.
Kamus aktif dan checksum diekspor agar hasil dapat direproduksi. Memakai versi resource berbeda
atau mengganti kamus bisa menghasilkan output berbeda.

In [38]:
def read_lexicon(path, variant='variant', canonical='canonical', **read_csv_options):
    """Adapter for Kamus Alay/Colloquial/Combined/custom CSV; no forced approval."""
    lex = pd.read_csv(path, dtype=str, keep_default_na=False, **read_csv_options)
    lex = lex.rename(columns={variant: 'variant', canonical: 'canonical'})
    if not {'variant', 'canonical'} <= set(lex):
        raise ValueError('Butuh kolom variant dan canonical, atau atur nama kolom adapter.')
    for c in ['variant', 'canonical']:
        lex[c] = lex[c].str.strip().str.casefold()
    for c, default in {'source': Path(path).name, 'category': 'slang', 'language': 'id',
                       'confidence': '', 'review_status': 'pending'}.items():
        if c not in lex:
            lex[c] = default
    return lex


def approved_slang(lex):
    multiple = lex.groupby('variant').canonical.nunique()
    eligible = lex.variant.map(multiple).eq(1) & lex.review_status.eq('approved')
    eligible &= ~lex.variant.isin(PROTECTED | AMBIGUOUS) & lex.variant.str.fullmatch(r'[^\W\d_]+')
    eligible &= lex.canonical.ne('')
    return {**SLANG, **dict(lex.loc[eligible, ['variant', 'canonical']].itertuples(index=False, name=None))}


RESOURCE_URLS = {
    'colloquial.csv': 'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv',
    'kamusalay.csv': 'https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/new_kamusalay.csv',
    'combined.json': 'https://raw.githubusercontent.com/louisowen6/NLP_bahasa_resources/master/combined_slang_words.txt',
}


def external_lexicons(folder, download=False):
    """Prepare online once; reuse offline. All community mappings remain pending.
    Original resource files stay local; review redistribution terms at source.
    """
    import json
    import urllib.request
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    tables = []
    for name, url in RESOURCE_URLS.items():
        path = folder / name
        if download and not path.exists():
            data = urllib.request.urlopen(url, timeout=45).read()
            tmp = path.with_suffix('.part')
            tmp.write_bytes(data)
            tmp.replace(path)
        if not path.exists():
            continue
        if name == 'combined.json':
            values = json.loads(path.read_text(encoding='utf-8'))
            table = pd.DataFrame(values.items(), columns=['variant', 'canonical'])
        elif name == 'kamusalay.csv':
            table = pd.read_csv(path, names=['variant', 'canonical'], encoding='latin-1', keep_default_na=False)
        else:
            table = pd.read_csv(path, keep_default_na=False).rename(columns={'slang': 'variant', 'formal': 'canonical'})
        table = table[['variant', 'canonical']].copy()
        table['source'] = url
        table['source_sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
        table['category'], table['language'] = 'slang', 'id'
        table['confidence'], table['review_status'] = '', 'pending'
        tables.append(table)
    if not tables:
        return pd.DataFrame(columns=['variant', 'canonical', 'source', 'source_sha256', 'category', 'language', 'confidence', 'review_status'])
    lex = pd.concat(tables, ignore_index=True).drop_duplicates(['variant', 'canonical', 'source'])
    for c in ['variant', 'canonical']:
        lex[c] = lex[c].str.strip().str.casefold()
    lex['conflict'] = lex.variant.map(lex.groupby('variant').canonical.nunique()).gt(1)
    return lex

In [39]:
lex = external_lexicons(RESOURCE_DIR, download=DOWNLOAD_RESOURCES)
OUT.mkdir(parents=True, exist_ok=True)
lex.to_csv(OUT / 'lexicon_candidates.csv', index=False)
if REVIEWED_LEXICON.exists():
    lex = pd.concat([lex, read_lexicon(REVIEWED_LEXICON)], ignore_index=True)
SLANG_USED = approved_slang(lex) if len(lex) else dict(SLANG)
print('Kandidat:', len(lex), '| mapping aktif:', len(SLANG_USED))
display(pd.DataFrame(SLANG_USED.items(), columns=['variant', 'canonical']).head(12))

Kandidat: 20595 | mapping aktif: 23


,variant,canonical
0,gk,tidak
1,ga,tidak
2,gak,tidak
3,nggak,tidak
4,ngga,tidak
5,tdk,tidak
6,blm,belum
7,jgn,jangan
8,yg,yang
9,dgn,dengan


### C1 · Normalisasi slang dan singkatan informal

Exact-token mapping, bukan replace substring. `gk,` tetap dikenali, isi URL/email/hashtag tidak diubah. `sm/dr/bs` tidak ditebak. Akronim MBG/SPPG/BGN tidak diekspansi otomatis.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [40]:
def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

In [41]:
texts = run_step(texts, lambda s: normalize_slang(s, SLANG_USED), 'C1 · Normalisasi slang dan singkatan informal', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('mbg bgs bgt, tp gk merata sm dr SPPG', lambda s: normalize_slang(s, SLANG_USED))

C1 · Normalisasi slang dan singkatan informal: aktif=True; berubah=19,230/57,362; kosong=485


,before,after
64,'anjir kebayangan kalo chattingan sama ibunya sekocak apa','anjir kebayangan kalau chattingan sama ibunya sekocak apa'
72,' user seru bgt',' user seru banget'
79,"'bubar aja dpr , jngan mbg sangat berterima kasih adanya mbg anak saya susah sarapan jd gk khawatir lg sekarang maka...","'bubar aja dpr , jngan mbg sangat berterima kasih adanya mbg anak saya susah sarapan jd tidak khawatir lg sekarang m..."
89,'iya gak enak','iya tidak enak'
91,'lagi serius nyimak emang bener ga usah buka komentar','lagi serius nyimak emang bener tidak usah buka komentar'
103,"'bukan times new roman lg ini, tp arial😭'","'bukan times new roman lg ini, tapi arial😭'"
117,"'salah yang masak sih kayak, soalnya mereka ngincar nya kan biara anak anak suka dan gak ada komplain'","'salah yang masak sih kayak, soalnya mereka ngincar nya kan biara anak anak suka dan tidak ada komplain'"
132,'penegasan istilah yg tidak berbelit😭🙏','penegasan istilah yang tidak berbelit😭🙏'


### C2 · Normalisasi huruf memanjang

Kurangi pengulangan 3+ hanya jika hasil ada di VOCAB. Double letter sah seperti massa/Allah tetap utuh; kata yang belum dikenal tidak dipaksa. Perluas VOCAB dari sumber yang dikurasi.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [42]:
def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

In [43]:
texts = run_step(texts, normalize_elongation, 'C2 · Normalisasi huruf memanjang', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('baguuuuus bangeeet Allah massa xyztttt', normalize_elongation)

C2 · Normalisasi huruf memanjang: aktif=True; berubah=207/57,362; kosong=485


,before,after
944,'iya bang seru bangettt + langsung mancep ke otak dan sangat mudah untuk dipahami','iya bang seru banget + langsung mancep ke otak dan sangat mudah untuk dipahami'
1121,'masya allaaaahhh 😍 semoga tetep begini.','masya allah 😍 semoga tetep begini.'
1174,'bagusss😭🔥🖤','bagus😭🔥🖤'
1216,'bagus bangeeet ka mbg nya. banyak dan bergizi banget. keliatannya juga enak ka. beruntung sekali anak2 dpt mbg seba...,'bagus banget ka mbg nya. banyak dan bergizi banget. keliatannya juga enak ka. beruntung sekali anak2 dpt mbg sebagu...
1259,'nahhhh ini terlihat enakkk','nahhhh ini terlihat enak'
1749,'aku udah pernah denger dr. tan ngomel tapi buat kebaikan jugaaa 😭','aku udah pernah denger dr. tan ngomel tapi buat kebaikan juga 😭'
1777,"'heh sayang dimana skolah yang gratisss, sebutin dongg'","'heh sayang dimana skolah yang gratis, sebutin dongg'"
1839,"'bagian mana sekolah gratissss,papua tidak ada namanya sekolah gratis.,'","'bagian mana sekolah gratis,papua tidak ada namanya sekolah gratis.,'"


### C3 · Normalisasi tawa

Kelompokkan variasi pola wkwk/haha yang eksplisit. Tawa tetap token ekspresi, tidak dibuang.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [44]:
def normalize_laughter(text):
    def replace(word):
        return 'wkwk' if re.fullmatch(r'(?:wk){2,}|w+k+w+k+[wk]*', word, re.I) else (
            'haha' if re.fullmatch(r'(?:ha){2,}h?', word, re.I) else word)
    return map_words(text, replace)

In [45]:
texts = run_step(texts, normalize_laughter, 'C3 · Normalisasi tawa', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('wkwkwkwkwk hahahaha bagus', normalize_laughter)

C3 · Normalisasi tawa: aktif=True; berubah=552/57,362; kosong=485


,before,after
21,'hahahahaha','haha'
44,'font style nya lebih ke arial sih. soalnya straight to the point hahaha','font style nya lebih ke arial sih. soalnya straight to the point haha'
51,'wkwkwk bener lagi','wkwk bener lagi'
77,'jelas wkwkw','jelas wkwk'
81,'wkwkwk😭','wkwk😭'
96,'wkwkwkwk analoginya ada aja abang satu ini 😂','wkwk analoginya ada aja abang satu ini 😂'
109,'hahaha','haha'
112,'wkwkkwkwkwkkwkwk','wkwk'


### C4 · Normalisasi reduplikasi angka 2

Whitelist `anak2/orang2/teman2`, bukan semua token berangka. S2/GPT2/angka tanggal tetap.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [46]:
def normalize_reduplication(text, mapping=None):
    # A whitelist avoids turning grade S2, model GPT2, dates or quantities into words.
    mapping = {'anak2': 'anak-anak', 'orang2': 'orang-orang', 'teman2': 'teman-teman'} if mapping is None else mapping
    return re.sub(r'\b\w+2\b', lambda m: mapping.get(m[0].casefold(), m[0]), text)

In [47]:
texts = run_step(texts, normalize_reduplication, 'C4 · Normalisasi reduplikasi angka 2', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('anak2 orang2 teman2 S2 GPT2', normalize_reduplication)

C4 · Normalisasi reduplikasi angka 2: aktif=True; berubah=871/57,362; kosong=485


,before,after
696,'makanya jadi tugas panitia untuk edukasi anak2 yang belum familiar dengan cuisine khas negaranya sendiri 😭 kenalkan...,'makanya jadi tugas panitia untuk edukasi anak-anak yang belum familiar dengan cuisine khas negaranya sendiri 😭 kena...
700,'tanya anak2 yang dapat saja gimana','tanya anak-anak yang dapat saja gimana'
704,'anak2 mana ngerti soal gizi malah diturutin😂','anak-anak mana ngerti soal gizi malah diturutin😂'
718,'iya emg request anak2. kata siibu nya juga kalau anak2 pesen cilok mau dibkin juga ? 😂😂','iya emg request anak-anak. kata siibu nya juga kalau anak-anak pesen cilok mau dibkin juga ? 😂😂'
765,"'ya masa harus nurutin request anak2. ini program makan bergizi. gacoan, burger mana ada gizinya. kenyang iya, enak ...","'ya masa harus nurutin request anak-anak. ini program makan bergizi. gacoan, burger mana ada gizinya. kenyang iya, e..."
771,'menu seperti itu hanya di hari jumat ajaa kalau di sekolah sini. g tiap hari dan memang yang paling di nanti sama a...,'menu seperti itu hanya di hari jumat ajaa kalau di sekolah sini. g tiap hari dan memang yang paling di nanti sama a...
931,"'kami masih bersyukur bu anak2 dpt mbg, menunya juga berganti2'","'kami masih bersyukur bu anak-anak dpt mbg, menunya juga berganti2'"
1019,"'klombok kl tidak pcy kak , anak2 kdg dpt lengkemg juga 🤭'","'klombok kl tidak pcy kak , anak-anak kdg dpt lengkemg juga 🤭'"


### C5 · Normalisasi bahasa daerah opsional

Dimatikan default. Contoh mapping kecil, bukan penerjemah. Aktifkan hanya setelah mengecek konteks; campuran bahasa tidak otomatis dianggap salah ketik.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [48]:
def normalize_regional(text, mapping=REGIONAL):
    return map_words(text, lambda w: mapping.get(w.casefold(), w))

In [49]:
texts = run_step(texts, normalize_regional, 'C5 · Normalisasi bahasa daerah opsional', enabled=NORMALIZE_REGIONAL)
if SHOW_RULE_EXAMPLES:
    show_example('ora apik dan teu enak', normalize_regional)

C5 · Normalisasi bahasa daerah opsional: aktif=False; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'","'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'"
1,'enter enter enter langsung ke inti nya','enter enter enter langsung ke inti nya'
2,'shift+enter langsung ke lembar berikutnya🗿','shift+enter langsung ke lembar berikutnya🗿'
3,' user ',' user '
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭ngakak','😭😭ngakak'
7,"'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'","'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'"


### C6 · Terapkan typo yang sudah direview

TYPO_APPROVED kosong secara default. Masukkan pasangan yang pasti setelah review. Kata tidak dikenal tidak otomatis dianggap typo. Nama lowercase masih bisa salah dikoreksi jika mapping sembarangan, sehingga review tetap diperlukan.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [50]:
def apply_reviewed_typos(text, mapping=None):
    mapping = {} if mapping is None else mapping
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

In [51]:
texts = run_step(texts, lambda s: apply_reviewed_typos(s, TYPO_APPROVED), 'C6 · Terapkan typo yang sudah direview', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('maknan bergzi SPPG', lambda s: apply_reviewed_typos(s, TYPO_APPROVED))

C6 · Terapkan typo yang sudah direview: aktif=True; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'","'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'"
1,'enter enter enter langsung ke inti nya','enter enter enter langsung ke inti nya'
2,'shift+enter langsung ke lembar berikutnya🗿','shift+enter langsung ke lembar berikutnya🗿'
3,' user ',' user '
4,'🤣🤣🤣😭😭','🤣🤣🤣😭😭'
5,'🤣🤣','🤣🤣'
6,'😭😭ngakak','😭😭ngakak'
7,"'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'","'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'"


Contoh buatan hanya tampil bila `SHOW_RULE_EXAMPLES=True`. Tanpa mapping typo yang sudah disetujui, teks dataset tetap utuh pada tahap ini.

In [52]:
if SHOW_RULE_EXAMPLES:
    show_example('maknan bergzi', lambda s: apply_reviewed_typos(s, {'maknan': 'makanan', 'bergzi': 'bergizi'}))

### C7 · Ringkas tanda baca berulang

Pengulangan ekstrem jadi maksimal dua: `!!!!!→!!`. Jenis tanda baca tidak disamakan. Cabang clean di bawah akan membuang tanda bacanya sepenuhnya.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [53]:
def normalize_punctuation_repeat(text):
    return re.sub(r'([!?.,])\1{2,}', r'\1\1', text)

In [54]:
texts = run_step(texts, normalize_punctuation_repeat, 'C7 · Ringkas tanda baca berulang', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('serius????!!!! bagus.....', normalize_punctuation_repeat)

C7 · Ringkas tanda baca berulang: aktif=True; berubah=3,681/57,362; kosong=485


,before,after
41,"'banyak yang tidak senang dengan ibu ini, karena dia berkata jujur...'","'banyak yang tidak senang dengan ibu ini, karena dia berkata jujur..'"
43,"'italic, bold, underline ...enter enter enter'","'italic, bold, underline ..enter enter enter'"
85,'jian tenan...🤣🤣','jian tenan..🤣🤣'
179,'print troooss...','print troooss..'
190,'oh my goddddd...','oh my goddddd..'
256,"'kayaknya 24pt arial kak bukan 16 lagi 😌,,,'","'kayaknya 24pt arial kak bukan 16 lagi 😌,,'"
276,'and then....shut down...😂😂','and then..shut down..😂😂'
285,'arial bold capital....berasa banget emosinya😫','arial bold capital..berasa banget emosinya😫'


In [55]:
texts = run_step(texts, normalize_whitespace, 'C8 · Rapikan spasi normalisasi')
text_normalized = texts.copy()

C8 · Rapikan spasi normalisasi: aktif=True; berubah=2,124/57,362; kosong=485


,before,after
3,' user ','user'
10,' user user ','user user'
13,' user 😭','user 😭'
15,' user bella','user bella'
22,' user user 😭','user user 😭'
24,'😭😭 user ','😭😭 user'
29,' user user user 😭','user user user 😭'
32,' user 😭','user 😭'


## D · Cleaning agresif: teks sebersih mungkin

Cabang ini mulai dari `text_normalized`. Emoji dan tanda baca dibuang, repetisi kata diringkas,
stopword/stemming dijalankan. **Informasi ekspresi dapat hilang**; before/after menunjukkan
persis yang berubah. Teks asli/safe/normalized tidak tertimpa. Tidak menghapus seluruh karakter
non-ASCII, tidak membuang kata pendek, dan tidak memotong komentar panjang diam-diam.

### D1 · Hapus emoji

Hapus satu sequence emoji secara utuh, termasuk emoji keluarga/skin tone/flag. Alternatif deskripsi emoji tersedia terpisah di bagian E.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [56]:
def remove_emoji(text):
    return emoji.replace_emoji(text, replace=' ')

In [57]:
texts = run_step(texts, remove_emoji, 'D1 · Hapus emoji', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('bagus 😍 🙏🏽 👨\u200d👩\u200d👧\u200d👦 🇮🇩', remove_emoji)

D1 · Hapus emoji: aktif=True; berubah=21,672/57,362; kosong=485


,before,after
2,'shift+enter langsung ke lembar berikutnya🗿','shift+enter langsung ke lembar berikutnya '
4,'🤣🤣🤣😭😭',' '
5,'🤣🤣',' '
6,'😭😭ngakak',' ngakak'
8,'🥰🥰🥰iya iya iya',' iya iya iya'
9,'😭',' '
11,'🤣😭',' '
12,'🤣',' '


### D2 · Hapus emoticon ASCII

Aturan umum :) :( :)) <3, bukan semua emoticon yang pernah ada. Tanda baca sisanya dibersihkan di D4.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [58]:
def remove_emoticons(text):
    return re.sub(r"(?<!\w)(?:[:;=8][\-^']?[)(/DPp]+|<3)(?!\w)", ' ', text)

In [59]:
texts = run_step(texts, remove_emoticons, 'D2 · Hapus emoticon ASCII', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('bagus :)) sedih :( cinta <3', remove_emoticons)

D2 · Hapus emoticon ASCII: aktif=True; berubah=32/57,362; kosong=485


,before,after
440,'@<3','@ '
805,'baru aja hotdog :(','baru aja hotdog '
2100,"'gw sekolah di swasta + negeri.. kalau swasta jelas byr kaya spp, uang bangunan, buku juga beli, seragam full beli, ...","'gw sekolah di swasta + negeri.. kalau swasta jelas byr kaya spp, uang bangunan, buku juga beli, seragam full beli, ..."
3670,'dia ahli gizi kasdi oon :)','dia ahli gizi kasdi oon '
3893,'ini kak bisa di check :)','ini kak bisa di check '
4093,"'sebenernya bukan pejabatnya aja yang harus di perbaiki , tapii sdm indonesia juga harus di perbaiki :(( keracunan m...","'sebenernya bukan pejabatnya aja yang harus di perbaiki , tapii sdm indonesia juga harus di perbaiki keracunan mak..."
4413,'tergantung vendornya :)))','tergantung vendornya '
4574,'sebenernya dari susu kotaknya aja udah tidak sehat karena kandungan gulanya itu lho besar sekali. jatuhnya malah ja...,'sebenernya dari susu kotaknya aja udah tidak sehat karena kandungan gulanya itu lho besar sekali. jatuhnya malah ja...


### D3 · Hapus angka opsional

Dimatikan default. Jika diaktifkan nominal/tanggal akan kehilangan angka; gunakan hanya jika memang tidak relevan pada task. Angka di kode alfanumerik tidak otomatis dihapus.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [60]:
def remove_numbers(text):
    # Opt-in destructive view: amounts/dates become incomplete, inspect before using.
    return re.sub(r'(?<!\w)\d+(?:[.,:/-]\d+)*(?!\w)', ' ', text)

In [61]:
texts = run_step(texts, remove_numbers, 'D3 · Hapus angka opsional', enabled=REMOVE_NUMBERS)
if SHOW_RULE_EXAMPLES:
    show_example('3 siswa Rp10.000 pada 10/11/2026 S2', remove_numbers)

D3 · Hapus angka opsional: aktif=False; berubah=0/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'","'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'"
1,'enter enter enter langsung ke inti nya','enter enter enter langsung ke inti nya'
2,'shift+enter langsung ke lembar berikutnya ','shift+enter langsung ke lembar berikutnya '
3,'user','user'
4,' ',' '
5,' ',' '
6,' ngakak',' ngakak'
7,"'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'","'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'"


### D4 · Hapus tanda baca dan simbol

Ubah ke spasi agar kata tidak menempel. Huruf Unicode/angka tetap. Nama dengan tanda hubung berubah menjadi kata terpisah; periksa implikasinya untuk entitas.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [62]:
def remove_punctuation(text):
    # Keep all scripts/letters/digits; punctuation becomes a SPACE, never concatenation.
    return re.sub(r'[^\w\s]|_', ' ', text)

In [63]:
texts = run_step(texts, remove_punctuation, 'D4 · Hapus tanda baca dan simbol', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('anak-anak, bagus!!! tidak/buruk; 10.000', remove_punctuation)

D4 · Hapus tanda baca dan simbol: aktif=True; berubah=27,007/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'",'ibuk ngomongnya times new roman bold caplock nyala 16 ctrl e'
2,'shift+enter langsung ke lembar berikutnya ','shift enter langsung ke lembar berikutnya '
7,"'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'",'sepertinya tombolnya bukan enter kak tapi entar jadi agak delay ke tujuan'
25,'@• アルタイル ',' アルタイル '
37,"'delete"" aja '",'delete aja '
38,'kasta beliau adlh diatas emak2 mpasi bang.. klu buka ig beliau brsa di marahi emak nya emak2','kasta beliau adlh diatas emak2 mpasi bang klu buka ig beliau brsa di marahi emak nya emak2'
41,"'banyak yang tidak senang dengan ibu ini, karena dia berkata jujur..'",'banyak yang tidak senang dengan ibu ini karena dia berkata jujur '
43,"'italic, bold, underline ..enter enter enter'",'italic bold underline enter enter enter'


### D5 · Ringkas repetisi kata berturut-turut

Hanya kata identik berurutan. Ekspresi `bagus bagus` akan jadi `bagus`; normalized masih menyimpan ekspresinya.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [64]:
def reduce_repeated_words(text):
    # Aggressive only: even "bagus bagus" can be expressive, so retain another view.
    return re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text, flags=re.I)

In [65]:
texts = run_step(texts, reduce_repeated_words, 'D5 · Ringkas repetisi kata berturut-turut', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('bagus bagus bagus tapi tidak tidak', reduce_repeated_words)

D5 · Ringkas repetisi kata berturut-turut: aktif=True; berubah=4,128/57,362; kosong=485


,before,after
1,'enter enter enter langsung ke inti nya','enter langsung ke inti nya'
8,' iya iya iya',' iya'
10,'user user','user'
16,'enter enter ','enter '
20,'esc esc esc kak ','esc kak '
22,'user user ','user '
29,'user user user ','user '
35,'user user','user'


### D6 · Hapus stopword

Daftar kecil yang transparan dan bisa diganti. Negasi dan akronim terlindungi tidak dibuang walaupun masuk daftar tambahan.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [66]:
def remove_stopwords(text, words=STOPWORDS):
    blocked = set(words) - NEGATIONS - PROTECTED
    return ' '.join(w for w in text.split() if w.casefold() not in blocked)

In [67]:
texts = run_step(texts, remove_stopwords, 'D6 · Hapus stopword', enabled=REMOVE_STOPWORDS)
if SHOW_RULE_EXAMPLES:
    show_example('makanan yang bagus untuk anak dan tidak merata', remove_stopwords)

D6 · Hapus stopword: aktif=True; berubah=46,080/57,362; kosong=2,958


,before,after
0,'ibuk ngomongnya times new roman bold caplock nyala 16 ctrl e','ibuk ngomongnya times new roman bold caplock nyala 16 ctrl e'
1,'enter langsung ke inti nya','enter langsung inti nya'
2,'shift enter langsung ke lembar berikutnya ','shift enter langsung lembar berikutnya'
4,' ',''
5,' ',''
6,' ngakak','ngakak'
7,'sepertinya tombolnya bukan enter kak tapi entar jadi agak delay ke tujuan','sepertinya tombolnya bukan enter kak tapi entar jadi agak delay tujuan'
8,' iya','iya'


### D7 · Stemming Sastrawi

Bentuk dasar per kata unik dengan cache terbatas. Negasi dan akronim terlindungi. Stemming bukan spellchecker dan dapat mengubah nama/istilah: safe/normalized tetap ada.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [68]:
@lru_cache(maxsize=1)
def get_stemmer():
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    return StemmerFactory().create_stemmer()


@lru_cache(maxsize=50000)
def stem_word(word):
    return get_stemmer().stem(word)


def stem_text(text):
    return map_words(text, lambda w: w if w.casefold() in PROTECTED | NEGATIONS else stem_word(w))

In [69]:
texts = run_step(texts, stem_text, 'D7 · Stemming Sastrawi', enabled=STEMMING)
if SHOW_RULE_EXAMPLES:
    show_example('makanan membantu pemerataan dan tidak bergizi SPPG', stem_text)

D7 · Stemming Sastrawi: aktif=True; berubah=30,389/57,362; kosong=2,969


,before,after
2,'shift enter langsung lembar berikutnya','shift enter langsung lembar ikut'
7,'sepertinya tombolnya bukan enter kak tapi entar jadi agak delay tujuan','seperti tombol bukan enter kak tapi entar jadi agak delay tuju'
25,'アルタイル',''
26,'tab nya kebanyakan kak','tab nya banyak kak'
27,'soalnya rata tengah','soal rata tengah'
38,'kasta beliau adlh diatas emak2 mpasi bang klu buka ig beliau brsa marahi emak nya emak2','kasta beliau adlh atas emak2 mpasi bang klu buka ig beliau brsa marah emak nya emak2'
41,'banyak tidak senang ibu karena berkata jujur','banyak tidak senang ibu karena kata jujur'
42,'1 lagi bang ratu rumpi jos nih didesa desa','1 lagi bang ratu rumpi jos nih desa desa'


In [70]:
texts = run_step(texts, normalize_whitespace, 'D8 · Rapikan hasil akhir')
text_clean = texts.copy()

D8 · Rapikan hasil akhir: aktif=True; berubah=115/57,362; kosong=3,003


,before,after
48,'ngakak bangett ore','ngakak bangett ore'
101,'user ','user'
277,'user ','user'
304,' mora user s raaa user','mora user s raaa user'
501,'user ','user'
550,'user ','user'
564,'user ','user'
569,'user ','user'


## E · Versi alternatif dan tokenisasi

Versi berikut **tidak menyambung cleaning agresif**: deskripsi emoji berangkat dari normalized,
IndoBERTweet berangkat dari safe. Tokenisasi untuk model pretrained harus memakai tokenizer
modelnya sendiri; `split()` di sini hanya tokenisasi sederhana hasil clean.

In [71]:
emoji_texts = text_normalized.copy()

### E1 · Emoji menjadi deskripsi

Shortcode Inggris dari library emoji, bukan hasil klasifikasi sentimen dan bukan terjemahan Indonesia.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [72]:
def describe_emoji(text):
    return emoji.demojize(text, delimiters=(' ', ' '))

In [73]:
emoji_texts = run_step(emoji_texts, describe_emoji, 'E1 · Emoji menjadi deskripsi', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('Menu 😍 👩\u200d🍳', describe_emoji)

E1 · Emoji menjadi deskripsi: aktif=True; berubah=21,672/57,362; kosong=485


,before,after
2,'shift+enter langsung ke lembar berikutnya🗿','shift+enter langsung ke lembar berikutnya moai '
4,'🤣🤣🤣😭😭',' rolling_on_the_floor_laughing rolling_on_the_floor_laughing rolling_on_the_floor_laughing loudly_crying_face l...
5,'🤣🤣',' rolling_on_the_floor_laughing rolling_on_the_floor_laughing '
6,'😭😭ngakak',' loudly_crying_face loudly_crying_face ngakak'
8,'🥰🥰🥰iya iya iya',' smiling_face_with_hearts smiling_face_with_hearts smiling_face_with_hearts iya iya iya'
9,'😭',' loudly_crying_face '
11,'🤣😭',' rolling_on_the_floor_laughing loudly_crying_face '
12,'🤣',' rolling_on_the_floor_laughing '


In [74]:
bert_texts = text_safe.copy()

### E2 · Resep khusus IndoBERTweet

Lowercase, mention→@USER, URL→HTTPURL, emoji shortcode mengikuti resep publik model. Jangan menambahkan stopword/stemming ke view ini tanpa eksperimen.

**Input:** hasil tahap sebelumnya. **Output:** teks tahap ini. Tabel memakai komentar asli dari CSV MBG. Jika tidak ada perubahan, tahap itu memang tidak menemukan pola yang perlu diganti pada data ini.

In [75]:
def indobertweet_view(text):
    text = URL.sub('HTTPURL', text.lower())
    text = re.sub(r'(?<![\w@])@[\w.]+', '@USER', text)
    return normalize_whitespace(emoji.demojize(text))

In [76]:
bert_texts = run_step(bert_texts, indobertweet_view, 'E2 · Resep khusus IndoBERTweet', enabled=True)
if SHOW_RULE_EXAMPLES:
    show_example('@Budi MBG BAGUS 😍 https://contoh.id', indobertweet_view)

E2 · Resep khusus IndoBERTweet: aktif=True; berubah=36,992/57,362; kosong=485


,before,after
0,"'ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E'","'ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e'"
1,'ENTER ENTER ENTER Langsung ke INTI nya','enter enter enter langsung ke inti nya'
2,'SHIFT+ENTER langsung ke lembar berikutnya🗿','shift+enter langsung ke lembar berikutnya:moai:'
3,'@Lfajeriah','@USER'
4,'🤣🤣🤣😭😭',':rolling_on_the_floor_laughing::rolling_on_the_floor_laughing::rolling_on_the_floor_laughing::loudly_crying_face::l...
5,'🤣🤣',':rolling_on_the_floor_laughing::rolling_on_the_floor_laughing:'
6,'😭😭Ngakak',':loudly_crying_face::loudly_crying_face:ngakak'
7,"'sepertinya tombolnya bukan ENTER kak, tapi ENTAR, jadi agak Delay ke tujuan'","'sepertinya tombolnya bukan enter kak, tapi entar, jadi agak delay ke tujuan'"


### E3 · Tokenisasi hasil clean

Before berupa kalimat, after berupa list kata. Bukan vocabulary/embedding, belum melatih apa pun.

In [77]:
def tokenize(text):
    # Transparent baseline tokenization of the clean view; not a model tokenizer.
    return text.split()

In [78]:
tokens = text_clean.map(tokenize)
display(pd.DataFrame({'before': text_clean, 'after_tokens': tokens}).head(PREVIEW_ROWS))
if SHOW_RULE_EXAMPLES:
    show_example('menu sehat tidak merata', tokenize)

,before,after_tokens
0,ibuk ngomongnya times new roman bold caplock nyala 16 ctrl e,"[ibuk, ngomongnya, times, new, roman, bold, caplock, nyala, 16, ctrl, e]"
1,enter langsung inti nya,"[enter, langsung, inti, nya]"
2,shift enter langsung lembar ikut,"[shift, enter, langsung, lembar, ikut]"
3,user,[user]
4,,[]
5,,[]
6,ngakak,[ngakak]
7,seperti tombol bukan enter kak tapi entar jadi agak delay tuju,"[seperti, tombol, bukan, enter, kak, tapi, entar, jadi, agak, delay, tuju]"


## F · Audit, pemakaian ulang, dan ekspor

Laporan menampilkan jumlah berubah/kosong tiap tahap. Baris kosong tetap dipertahankan;
filter hanya setelah mengetahui aturan dataset/task. Tidak ada klaim skor meningkat karena
teks menjadi lebih pendek atau jumlah kata unik turun.

In [79]:
result = df.copy()
for name in ['text_raw', 'text_safe', 'text_normalized', 'text_clean', 'text_emoji_described', 'text_indobertweet', 'tokens']:
    if name in result:
        raise ValueError(f'Kolom output {name} sudah ada. Gunakan input mentah.')
result['text_raw'], result['text_safe'] = raw, text_safe
result['text_normalized'], result['text_clean'] = text_normalized, text_clean
result['text_emoji_described'], result['text_indobertweet'] = emoji_texts, bert_texts
result['tokens'] = tokens
audit_table = pd.DataFrame(audit)
display(audit_table)
display(result[['text_raw', 'text_safe', 'text_normalized', 'text_clean']])
# Pilih baris untuk menelusuri seluruh transformasinya (nomor POSISI, bukan label index).
ROW = min(6, len(result)-1)
if ROW >= 0:
    display(pd.DataFrame([(label, visible(values.iloc[ROW])) for label, values in history.items()],
                         columns=['tahap', 'hasil_satu_komentar']))

,stage,enabled,changed,empty,seconds
0,A1 · Validasi tipe teks,True,485,485,0.0855
1,A2 · Decode byte escape,False,0,485,0.0743
2,A3 · Perbaiki encoding rusak,True,0,485,0.8131
3,A4 · Normalisasi Unicode NFC,True,2,485,0.1168
4,A5 · Buang markup HTML bila ada,False,0,485,0.0810
5,A6 · Decode HTML entities,True,0,485,0.1188
6,A7 · Bersihkan karakter kontrol,True,0,485,0.3275
7,A8 · Rapikan whitespace,True,2888,485,0.2798
8,B1 · Normalisasi teks tersamarkan,True,0,485,0.3349
9,B2 · Ganti URL,True,22,485,0.2093


,text_raw,text_safe,text_normalized,text_clean
0,"ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E","ibuk ngomongnya, TIMES NEW ROMAN, BOLD, CAPLOCK NYALA, 16, CTRL +E","ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e",ibuk ngomongnya times new roman bold caplock nyala 16 ctrl e
1,ENTER ENTER ENTER Langsung ke INTI nya,ENTER ENTER ENTER Langsung ke INTI nya,enter enter enter langsung ke inti nya,enter langsung inti nya
2,SHIFT+ENTER langsung ke lembar berikutnya🗿,SHIFT+ENTER langsung ke lembar berikutnya🗿,shift+enter langsung ke lembar berikutnya🗿,shift enter langsung lembar ikut
3,@Lfajeriah,@Lfajeriah,user,user
4,🤣🤣🤣😭😭,🤣🤣🤣😭😭,🤣🤣🤣😭😭,
...,...,...,...,...
57357,gas,gas,gas,gas
57358,kepala BGN juga,kepala BGN juga,kepala bgn juga,kepala bgn
57359,komentarnya sepi 🔥🔥🔥,komentarnya sepi 🔥🔥🔥,komentarnya sepi 🔥🔥🔥,komentar sepi
57360,🇲🇨💪🇲🇨💪🇲🇨,🇲🇨💪🇲🇨💪🇲🇨,🇲🇨💪🇲🇨💪🇲🇨,


,tahap,hasil_satu_komentar
0,A1 · Validasi tipe teks,'😭😭Ngakak'
1,A2 · Decode byte escape,'😭😭Ngakak'
2,A3 · Perbaiki encoding rusak,'😭😭Ngakak'
3,A4 · Normalisasi Unicode NFC,'😭😭Ngakak'
4,A5 · Buang markup HTML bila ada,'😭😭Ngakak'
5,A6 · Decode HTML entities,'😭😭Ngakak'
6,A7 · Bersihkan karakter kontrol,'😭😭Ngakak'
7,A8 · Rapikan whitespace,'😭😭Ngakak'
8,B1 · Normalisasi teks tersamarkan,'😭😭Ngakak'
9,B2 · Ganti URL,'😭😭Ngakak'


### F1 · Sekali panggil, hasil identik dengan sel bertahap

Bagian ini hanya menyusun fungsi-fungsi yang sudah dipelajari, tanpa menulis ulang aturan.
`preprocess_texts` memakai urutan dan mapping yang sama. Verifikasi otomatis di bawah membandingkan
semua output, termasuk urutan/index dan token. Untuk menyalin seluruh pipeline gunakan
`stepwise.py`; untuk satu langkah cukup salin fungsi dan dependensi yang ditampilkan pada selnya.

In [80]:
DEFAULT_OPTIONS = dict(html_input=False, escaped_bytes=False, regional=False,
                       remove_numbers=False, stem=True, stopwords=True,
                       slang=None, typos=None, hashtags=None)


def build_steps(options=None):
    unknown = set(options or {}) - set(DEFAULT_OPTIONS)
    if unknown:
        raise ValueError(f'Opsi tidak dikenal: {sorted(unknown)}')
    options = {**DEFAULT_OPTIONS, **(options or {})}
    safe = [('valid', valid_text)]
    if options['escaped_bytes']:
        safe.append(('escaped_bytes', decode_byte_escapes))
    safe += [('encoding', repair_encoding), ('unicode', normalize_unicode)]
    if options['html_input']:
        safe.append(('html', remove_html))
    safe += [('entities', decode_html_entities), ('invisible', remove_invisible), ('space', normalize_whitespace)]
    normalized = [('obfuscation', normalize_obfuscation), ('urls', replace_urls),
                  ('emails', replace_emails), ('phones', replace_phones), ('mentions', replace_mentions),
                  ('hashtags', lambda s: segment_hashtags(s, options['hashtags'])), ('lowercase', lowercase),
                  ('slang', lambda s: normalize_slang(s, SLANG if options['slang'] is None else options['slang'])),
                  ('elongation', normalize_elongation), ('laughter', normalize_laughter),
                  ('reduplication', normalize_reduplication)]
    if options['regional']:
        normalized.append(('regional', normalize_regional))
    normalized += [('typo', lambda s: apply_reviewed_typos(s, options['typos'])),
                   ('punctuation_repeat', normalize_punctuation_repeat), ('space', normalize_whitespace)]
    clean = [('emoji', remove_emoji), ('emoticons', remove_emoticons)]
    if options['remove_numbers']:
        clean.append(('numbers', remove_numbers))
    clean += [('punctuation', remove_punctuation), ('repeated_words', reduce_repeated_words)]
    if options['stopwords']:
        clean.append(('stopwords', remove_stopwords))
    if options['stem']:
        clean.append(('stemming', stem_text))
    clean.append(('space', normalize_whitespace))
    return dict(safe=safe, normalized=normalized, clean=clean)


def transform_unique(texts, function):
    # Invalid values handled separately, so unhashable input never becomes a cache key.
    texts = texts.map(valid_text) if function is valid_text else texts
    cache = {text: function(text) for text in pd.unique(texts)}
    return texts.map(cache)


def preprocess_texts(texts, options=None):
    """Same functions and order as notebook. No data-dependent fitting or randomness."""
    texts = texts.copy() if isinstance(texts, pd.Series) else pd.Series(texts)
    output = pd.DataFrame({'text_raw': texts})
    current = texts
    for branch, steps in build_steps(options).items():
        for _, function in steps:
            current = transform_unique(current, function)
        output['text_' + branch] = current
    output['text_emoji_described'] = output.text_normalized.map(describe_emoji)
    output['tokens'] = output.text_clean.map(tokenize)
    output['text_indobertweet'] = output.text_safe.map(indobertweet_view)
    return output

In [81]:
OPTIONS = dict(html_input=HTML_INPUT, escaped_bytes=DECODE_ESCAPED_BYTES,
    regional=NORMALIZE_REGIONAL, remove_numbers=REMOVE_NUMBERS,
    stopwords=REMOVE_STOPWORDS, stem=STEMMING, slang=SLANG_USED,
    typos=TYPO_APPROVED, hashtags=HASHTAG_APPROVED)
reused = preprocess_texts(raw, options=OPTIONS)
for col in reused:
    pd.testing.assert_series_equal(result[col], reused[col], check_names=False)
pd.testing.assert_series_equal(df[TEXT], raw)
assert len(result) == len(df)
print('LULUS: hasil step-by-step = API sekali panggil; raw/urutan/jumlah baris utuh.')

LULUS: hasil step-by-step = API sekali panggil; raw/urutan/jumlah baris utuh.


### F2 · Simpan hasil dan aturan yang digunakan

CSV untuk inspeksi, JSONL gzip untuk struktur token, `audit_stages.csv` untuk ringkasan,
`before_after_stages.csv.gz` untuk seluruh tahap dalam format lebar. Tabel lengkap tahap bersifat
opsional untuk data besar karena menggandakan penyimpanan; matikan `SAVE_ALL_STAGES`.
Manifest menyimpan versi library, konfigurasi dan mapping aktual (termasuk typo/hashtag).
Hasil identik membutuhkan input, urutan aturan, resource, dan versi library yang sama.

In [82]:
import importlib.metadata
SAVE_ALL_STAGES = True  # False untuk data besar
OUT.mkdir(parents=True, exist_ok=True)
result.to_json(OUT / 'comments.jsonl.gz', orient='records', lines=True, force_ascii=False)
csv = result.copy()
csv['tokens'] = csv.tokens.map(lambda v: json.dumps(v, ensure_ascii=False))
csv.to_csv(OUT / 'comments.csv', index=False, encoding='utf-8-sig')
audit_table.to_csv(OUT / 'audit_stages.csv', index=False)
if SAVE_ALL_STAGES:
    pd.DataFrame(history).to_csv(OUT / 'before_after_stages.csv.gz', index=False, encoding='utf-8-sig')
manifest = dict(rows=len(result), input_file=str(INPUT.resolve()), input_sha256=hashlib.sha256(INPUT.read_bytes()).hexdigest(), text_column=TEXT, sampled=False, options=OPTIONS,
    versions={m: importlib.metadata.version(m) for m in ['pandas', 'ftfy', 'emoji', 'Sastrawi', 'beautifulsoup4']},
    dictionaries=dict(slang=SLANG_USED, vocab=sorted(VOCAB), stopwords=sorted(STOPWORDS),
                      protected=sorted(PROTECTED), negations=sorted(NEGATIONS),
                      regional=REGIONAL, obfuscation=OBFUSCATION),
    resources={str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in RESOURCE_DIR.glob('*') if p.is_file()},
    stage_order=[a['stage'] for a in audit],
    selected_view='text_clean', note='text-only; no row deletion; no task accuracy benchmark')
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Hasil:', OUT.resolve())

Hasil: D:\Lomba\2026\Gemastik\Notebook\Preprocessing_universal\hasil_stepwise


### F3 · Data besar dan cara memilih view

Untuk ratusan ribu komentar, matikan preview/penyimpanan semua tahap atau proses CSV per batch.
Notebook ini sengaja menyimpan snapshot tiap tahap agar mudah dipelajari; mode produksi gunakan
API `stepwise.preprocess_texts` yang menyimpan hanya keluaran cabang. Kata identik diproses sekali
per tahap/batch. Stemming memakai cache per kata; pembentukan vocab dari data baru tidak otomatis.

```python
from stepwise import preprocess_texts
for i, chunk in enumerate(pd.read_csv('komentar.csv', dtype='string', chunksize=20000)):
    out = preprocess_texts(chunk['comment'], options=OPTIONS)
    out.to_json(f'hasil-{i:04d}.jsonl', orient='records', lines=True, force_ascii=False)
```

Untuk membandingkan performa model: evaluasi raw/safe/normalized/clean dengan fold, model,
seed, dan metrik yang sama. Stemming/stopword atau menghapus emoji tidak selalu membantu.
NER/QA harus menggunakan offset anotasi yang sesuai; cleaned text tidak otomatis selaras raw.
Kamus daerah/typo yang belum tervalidasi lebih baik tetap utuh daripada diubah berdasarkan tebakan.

## Referensi dan sumber tambahan

Ditinjau 8 November 2026. Blueprint pengguna menjadi referensi kebutuhan; aturan notebook
membedakan cleaning aman, normalisasi, dan penghapusan agresif agar perubahan dapat diperiksa.

| Sumber primer | Pemakaian |
|---|---|
| [ftfy](https://ftfy.readthedocs.io/en/latest/config.html) | Repair encoding; transformasi lain dikendalikan eksplisit |
| [emoji](https://emoji-python.readthedocs.io/en/latest/api.html) | Hapus sequence utuh atau deskripsi sebagai view alternatif |
| [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) | Parser markup opt-in, bukan regex HTML |
| [Sastrawi](https://github.com/har07/PySastrawi) | Stemming per kata dengan cache pada cabang clean |
| [IndoBERTweet](https://github.com/indolem/IndoBERTweet) | Resep model: lowercase, @USER, HTTPURL, emoji shortcode |
| [indoNLP](https://hyuto.github.io/indo-nlp/api/preprocessing/) | Referensi normalisasi slang/elongation; fungsi lokal dibuat transparan |
| [Colloquial Lexicon](https://github.com/nasalsabila/kamus-alay) | Adapter CSV slang/formal, mapping luar butuh review |
| [Kamus Alay](https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection) | Adapter CSV tanpa header; konflik tidak diterapkan otomatis |
| [Combined Slang](https://github.com/louisowen6/NLP_bahasa_resources) | Adapter JSON dan pencatatan sumber/hash |

Tambahan di luar blueprint: [IndoCollex](https://aclanthology.org/2021.findings-acl.280/)
sebagai benchmark normalisasi informal, dan [normalisasi code-mixed Indonesia–Inggris](https://aclanthology.org/D19-5554/)
untuk mengevaluasi campuran bahasa secara terpisah. Ini rujukan eksperimen, bukan jaminan skor MBG.

Sumber opsional untuk pengembangan: [SALTIK](https://github.com/ir-nlp-csui/saltik),
[regional lexicon](https://data.mendeley.com/datasets/mrfc9mb2yt/1),
[parallel normalization corpus](https://github.com/ir-nlp-csui/parallel-corpus-for-lexical-normalization),
[BASA](https://github.com/Muanai/basa), dan [SymSpell](https://symspellpy.readthedocs.io/en/latest/examples/lookup.html).
Tidak dianggap sebagai normalizer siap pakai tanpa evaluasi; kosakata regional dan nama sering ambigu.

Fungsi non-teks pada API lama di preprocessing.py memakai
[datasketch](https://ekzhu.com/datasketch/lsh.html),
[RapidFuzz](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html), dan
[fastText](https://github.com/facebookresearch/fastText/blob/main/docs/language-identification.md).
Ketiganya tidak dijalankan pada alur notebook teks yang baru.

Kamus asli tidak dibundel untuk redistribusi. URL, SHA-256, mapping aktif, config, dan versi
library dicatat lokal. Periksa ketentuan sumber sebelum mendistribusikan resource.
Ablation pada data berlabel dan validasi yang sama diperlukan sebelum menyatakan skor meningkat.


**Catatan versi step-by-step:** notebook ini fokus pada transformasi teks; modul typo-candidate/fastText/near-duplicate yang disebut pada referensi tetap tersedia di `legacy/preprocessing.py`, tetapi tidak dijalankan oleh notebook teks ini.